In [1]:
from IAA import batch_evaluate_folder

In [67]:
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
human_folder = fr"{project_root}\data\final\Annotated"
llm_folder = fr"{project_root}\data\Documents_Annotés\llm\p2_c500_fsselected-30_mgpt-5.2"

# Evaluation settings
evaluation_level = "both"  # Options: "level1", "level2", "both"
match_type = "context"     # Options: "context", "context_overlap"
context_chars = 200
version = "v1.3"           # Version string that LLM filenames must end with

# Run batch evaluation
all_results = batch_evaluate_folder(human_folder, llm_folder, evaluation_level, match_type, context_chars, version)


╔══════════════════════════════════════════════════════════════════════════════════════════════════╗
║                              BATCH EVALUATION MODE                                               ║
╚══════════════════════════════════════════════════════════════════════════════════════════════════╝

📁 Human Annotations: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\final\Annotated
📁 LLM Annotations:   C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\p2_c500_fsselected-30_mgpt-5.2
📄 Output Log:        C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\p2_c500_fsselected-30_mgpt-5.2\evaluation_results.txt

🔍 Found 6 human annotated files

────────────────────────────────────────────────────────────────────────────────────────────────────
[1/6] Processing: 1989CanLII1415ONCA_annotated_GL_tech.html
──────────────────────────────────────────────────────────────────────────────────

In [68]:
results = all_results[:6]

In [69]:
def build_match_maps(spans1, spans2):
    """
    Returns:
      exact_pairs: list of (i, j)
      lenient_pairs: list of (i, j) EXCLUDING exact matches
    """
    exact_pairs = []
    used_j_exact = set()

    # Exact matches
    for i, s1 in enumerate(spans1):
        for j, s2 in enumerate(spans2):
            if j not in used_j_exact and s1.context_match(s2):
                exact_pairs.append((i, j))
                used_j_exact.add(j)
                break

    exact_i = {i for i, _ in exact_pairs}
    exact_j = {j for _, j in exact_pairs}

    # Lenient-only matches
    lenient_pairs = []
    for i, s1 in enumerate(spans1):
        if i in exact_i:
            continue
        for j, s2 in enumerate(spans2):
            if j in exact_j:
                continue
            if s1.context_overlap(s2):
                lenient_pairs.append((i, j))

    return exact_pairs, lenient_pairs

In [70]:
def build_side_by_side_view(spans1, spans2):
    """
    Returns rows of:
    (span1_text | span2_text | match_type)
    """
    exact_pairs, lenient_pairs = build_match_maps(spans1, spans2)

    exact_i = {i for i, _ in exact_pairs}
    exact_j = {j for _, j in exact_pairs}

    # Map lenient matches (one-to-one, greedy, order-preserving)
    lenient_map_1 = {}
    lenient_map_2 = {}

    for i, j in lenient_pairs:
        if i not in lenient_map_1 and j not in lenient_map_2:
            lenient_map_1[i] = j
            lenient_map_2[j] = i

    rows = []
    used_j = set()

    # Walk annotator 1 in order
    for i, s1 in enumerate(spans1):
        if i in exact_i:
            continue  # skip exact matches

        if i in lenient_map_1:
            j = lenient_map_1[i]
            rows.append((s1.text, spans2[j].text, "≈ lenient"))
            used_j.add(j)
        else:
            rows.append((s1.text, "", "✗ no match"))

    # Remaining annotator 2 spans
    for j, s2 in enumerate(spans2):
        if j in exact_j or j in used_j:
            continue
        rows.append(("", s2.text, "✗ no match"))

    return rows

In [71]:
import textwrap

def print_side_by_side(rows, width=60):
    sep = " | "

    header = f"{'Annotator 1':<{width}}{sep}{'Annotator 2':<{width}}{sep}Match"
    print(header)
    print("=" * len(header))

    for a1, a2, tag in rows:
        # Wrap text into multiple lines
        a1_lines = textwrap.wrap(a1, width=width) if a1 else [""]
        a2_lines = textwrap.wrap(a2, width=width) if a2 else [""]

        max_lines = max(len(a1_lines), len(a2_lines))

        # Pad shorter one
        a1_lines += [""] * (max_lines - len(a1_lines))
        a2_lines += [""] * (max_lines - len(a2_lines))

        # Print aligned lines
        for l1, l2 in zip(a1_lines, a2_lines):
            print(f"{l1:<{width}}{sep}{l2:<{width}}{sep}{tag}")

        # Separator between span pairs
        print("-" * len(header))


import textwrap

def get_side_by_side(rows, width=60):
    sep = " | "
    header = f"{'Annotator 1':<{width}}{sep}{'Annotator 2':<{width}}{sep}Match"
    output = [header, "=" * len(header)]

    for a1, a2, tag in rows:
        a1_lines = textwrap.wrap(a1, width=width) if a1 else [""]
        a2_lines = textwrap.wrap(a2, width=width) if a2 else [""]

        max_lines = max(len(a1_lines), len(a2_lines))
        a1_lines += [""] * (max_lines - len(a1_lines))
        a2_lines += [""] * (max_lines - len(a2_lines))

        for l1, l2 in zip(a1_lines, a2_lines):
            output.append(f"{l1:<{width}}{sep}{l2:<{width}}{sep}{tag}")

        output.append("-" * len(header))

    return "\n".join(output)


### PDF Generation Report

In [21]:
# Install reportlab if not already installed
#!pip install reportlab


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
from reportlab.lib.pagesizes import letter, A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, PageBreak, Spacer
from reportlab.lib import colors
from reportlab.lib.enums import TA_LEFT
from datetime import datetime
import os

In [23]:
def generate_comparison_pdf(all_results, output_path="annotation_comparison.pdf"):
    """
    Generate a PDF with side-by-side comparison of human vs LLM annotations.
    
    Args:
        all_results: List of result dictionaries containing 'human_spans' and 'llm_spans'
        output_path: Path where the PDF will be saved
    """
    doc = SimpleDocTemplate(output_path, pagesize=letter,
                           rightMargin=0.5*inch, leftMargin=0.5*inch,
                           topMargin=0.5*inch, bottomMargin=0.5*inch)
    
    # Container for the 'Flowable' objects
    elements = []
    
    # Define styles
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontSize=16,
        textColor=colors.HexColor('#2C3E50'),
        spaceAfter=12,
        alignment=TA_LEFT
    )
    
    subtitle_style = ParagraphStyle(
        'CustomSubtitle',
        parent=styles['Heading2'],
        fontSize=12,
        textColor=colors.HexColor('#34495E'),
        spaceAfter=8,
        alignment=TA_LEFT
    )
    
    # Add title page
    elements.append(Paragraph("Human vs LLM Annotation Comparison Report", title_style))
    elements.append(Paragraph(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", styles['Normal']))
    elements.append(Spacer(1, 0.3*inch))
    elements.append(Paragraph(f"Total Documents Compared: {len(all_results)}", styles['Normal']))
    elements.append(Spacer(1, 0.5*inch))
    
    # Process each result
    for idx, result in enumerate(all_results, 1):
        # Get document name if available
        doc_name = result.get('document_name', f'Document {idx}')
        
        # Add document header
        elements.append(PageBreak())
        elements.append(Paragraph(f"Document {idx}: {doc_name}", title_style))
        elements.append(Spacer(1, 0.2*inch))
        
        # Get spans
        human_spans = result.get('human_spans', [])
        llm_spans = result.get('llm_spans', [])
        
        # Add summary statistics
        elements.append(Paragraph(f"Human Annotations: {len(human_spans)}", styles['Normal']))
        elements.append(Paragraph(f"LLM Annotations: {len(llm_spans)}", styles['Normal']))
        elements.append(Spacer(1, 0.2*inch))
        
        # Build side-by-side view
        rows = build_side_by_side_view(human_spans, llm_spans)
        
        if not rows:
            elements.append(Paragraph("No annotations to compare.", styles['Normal']))
            continue
        
        # Create table data
        table_data = [['Human Annotation', 'LLM Annotation', 'Match Status']]
        
        for human_text, llm_text, match_type in rows:
            # Truncate long texts for readability
            max_length = 200
            human_display = human_text[:max_length] + "..." if len(human_text) > max_length else human_text
            llm_display = llm_text[:max_length] + "..." if len(llm_text) > max_length else llm_text
            
            table_data.append([
                Paragraph(human_display or "(no match)", styles['Normal']),
                Paragraph(llm_display or "(no match)", styles['Normal']),
                match_type
            ])
        
        # Create table with styling
        col_widths = [3*inch, 3*inch, 1*inch]
        table = Table(table_data, colWidths=col_widths, repeatRows=1)
        
        # Define table style
        table_style = TableStyle([
            # Header styling
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#3498DB')),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('ALIGN', (0, 0), (-1, 0), 'CENTER'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, 0), 10),
            ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
            
            # Body styling
            ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
            ('TEXTCOLOR', (0, 1), (-1, -1), colors.black),
            ('ALIGN', (0, 1), (1, -1), 'LEFT'),
            ('ALIGN', (2, 1), (2, -1), 'CENTER'),
            ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
            ('FONTSIZE', (0, 1), (-1, -1), 8),
            ('TOPPADDING', (0, 1), (-1, -1), 6),
            ('BOTTOMPADDING', (0, 1), (-1, -1), 6),
            ('VALIGN', (0, 1), (-1, -1), 'TOP'),
            
            # Grid
            ('GRID', (0, 0), (-1, -1), 1, colors.grey),
            
            # Alternating row colors
            ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#ECF0F1')]),
        ])
        
        table.setStyle(table_style)
        elements.append(table)
        elements.append(Spacer(1, 0.3*inch))
    
    # Build PDF
    doc.build(elements)
    print(f"PDF generated successfully: {output_path}")
    return output_path

In [26]:
# Generate the PDF comparison report
output_pdf_path = fr"{project_root}\annotation_comparison_report.pdf"
generate_comparison_pdf(results, output_pdf_path)

PDF generated successfully: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\annotation_comparison_report.pdf


'C:\\Users\\zakga\\OneDrive\\Documents\\code\\LeREaD_annotation_process\\annotation_comparison_report.pdf'

### Interactive HTML Report Generation

In [72]:
import json
import html as html_escape

def build_comparison_data(all_results):
    """
    Build structured data for HTML report from all_results.
    Returns list of comparison rows with document, label, and span info.
    """
    comparison_data = []
    
    for idx, result in enumerate(all_results):
        doc_name = result.get('human_file', result.get('document_name', f'Document_{idx+1}'))
        human_spans = result.get('human_spans', [])
        llm_spans = result.get('llm_spans', [])
        
        # Build match maps
        exact_pairs, lenient_pairs = build_match_maps(human_spans, llm_spans)
        exact_i = {i for i, _ in exact_pairs}
        exact_j = {j for _, j in exact_pairs}
        
        # Create mapping for lenient matches
        lenient_map_1 = {}
        lenient_map_2 = {}
        for i, j in lenient_pairs:
            if i not in lenient_map_1 and j not in lenient_map_2:
                lenient_map_1[i] = j
                lenient_map_2[j] = i
        
        used_j = set()
        
        # Process exact matches first
        for i, j in exact_pairs:
            s1 = human_spans[i]
            s2 = llm_spans[j]
            comparison_data.append({
                'document': doc_name,
                'label_name': s1.labelname,
                'human_text': s1.text,
                'llm_text': s2.text,
                'match_type': '✓ exact',
                'comment': ''
            })
            used_j.add(j)
        
        # Process lenient matches
        for i, s1 in enumerate(human_spans):
            if i in exact_i:
                continue  # Already processed
            
            human_text = s1.text
            label_name = s1.labelname
            llm_text = ""
            match_type = "✗ no match"
            
            if i in lenient_map_1:
                j = lenient_map_1[i]
                llm_text = llm_spans[j].text
                match_type = "≈ lenient"
                used_j.add(j)
            
            comparison_data.append({
                'document': doc_name,
                'label_name': label_name,
                'human_text': human_text,
                'llm_text': llm_text,
                'match_type': match_type,
                'comment': ''
            })
        
        # Remaining LLM spans (not matched)
        for j, s2 in enumerate(llm_spans):
            if j in used_j:
                continue
            comparison_data.append({
                'document': doc_name,
                'label_name': s2.labelname,
                'human_text': '',
                'llm_text': s2.text,
                'match_type': '✗ no match',
                'comment': ''
            })
    
    return comparison_data

In [73]:
def generate_html_report(all_results, output_path="annotation_comparison.html"):
    """
    Generate an interactive HTML report with filters and color highlighting.
    
    Args:
        all_results: List of result dictionaries containing 'human_spans' and 'llm_spans'
        output_path: Path where the HTML will be saved
    """
    comparison_data = build_comparison_data(all_results)
    
    # Extract unique document names and labels for filters
    documents = sorted(set(row['document'] for row in comparison_data))
    labels = sorted(set(row['label_name'] for row in comparison_data if row['label_name']))
    
    # Convert data to JSON for JavaScript
    data_json = json.dumps(comparison_data, ensure_ascii=False)
    
    html_content = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Annotation Comparison Report</title>
    <style>
        * {{ box-sizing: border-box; margin: 0; padding: 0; }}
        body {{ font-family: Arial, sans-serif; padding: 20px; background: #f5f5f5; }}
        h1 {{ margin-bottom: 20px; color: #333; }}
        .controls {{ background: white; padding: 15px; margin-bottom: 20px; border-radius: 4px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); display: flex; justify-content: space-between; align-items: center; }}
        .controls button {{ padding: 8px 16px; margin-left: 10px; border: none; border-radius: 4px; cursor: pointer; font-weight: bold; }}
        .btn-export {{ background: #2196F3; color: white; }}
        .btn-export:hover {{ background: #1976D2; }}
        .btn-import {{ background: #4CAF50; color: white; }}
        .btn-import:hover {{ background: #45a049; }}
        .filters {{ background: white; padding: 15px; margin-bottom: 20px; border-radius: 4px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }}
        .filters label {{ margin-right: 10px; font-weight: bold; }}
        .filters select {{ padding: 5px; margin-right: 20px; border: 1px solid #ddd; border-radius: 3px; }}
        table {{ width: 100%; border-collapse: collapse; background: white; box-shadow: 0 2px 4px rgba(0,0,0,0.1); table-layout: fixed; }}
        th {{ background: #4CAF50; color: white; padding: 12px; text-align: left; position: sticky; top: 0; z-index: 10; }}
        td {{ padding: 10px; border-bottom: 1px solid #ddd; vertical-align: top; word-wrap: break-word; }}
        td:nth-child(1) {{ width: 10%; }}
        td:nth-child(2) {{ width: 10%; }}
        td:nth-child(3) {{ width: 25%; }}
        td:nth-child(4) {{ width: 25%; }}
        td:nth-child(5) {{ width: 8%; }}
        td:nth-child(6) {{ width: 10%; }}
        td:nth-child(7) {{ width: 12%; }}
        .comment-input {{ width: 100%; padding: 5px; border: 1px solid #ddd; border-radius: 3px; min-height: 40px; resize: vertical; }}
        .color-0 {{ background: #ffffff; }}
        .color-1 {{ background: #ffebee; }}
        .color-2 {{ background: #e8f5e9; }}
        .color-3 {{ background: #e3f2fd; }}
        .color-4 {{ background: #fff3e0; }}
        .color-5 {{ background: #f3e5f5; }}
        .color-6 {{ background: #fff9c4; }}
        .color-7 {{ background: #e0f2f1; }}
        .color-8 {{ background: #fce4ec; }}
        .color-9 {{ background: #e8eaf6; }}
        .stats {{ background: white; padding: 15px; margin-bottom: 20px; border-radius: 4px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }}
        .match-exact {{ color: #2e7d32; font-weight: bold; }}
        .match-lenient {{ color: #f57c00; }}
        .match-none {{ color: #c62828; }}
        #fileInput {{ display: none; }}
        .save-status {{ margin-left: 10px; color: #666; font-style: italic; }}
    </style>
</head>
<body>
    <h1>Human vs LLM Annotation Comparison</h1>
    
    <div class="controls">
        <div class="stats">
            <strong>Total Comparisons:</strong> <span id="total-count">0</span> |
            <strong>Visible:</strong> <span id="visible-count">0</span>
            <span class="save-status" id="save-status"></span>
        </div>
        <div>
            <button class="btn-export" onclick="exportData()">📥 Export Data</button>
            <button class="btn-import" onclick="document.getElementById('fileInput').click()">📤 Import Data</button>
            <input type="file" id="fileInput" accept=".json" onchange="importData(event)">
        </div>
    </div>
    
    <div class="filters">
        <label>Document:</label>
        <select id="doc-filter" onchange="applyFilters()">
            <option value="">All Documents</option>
            {''.join(f'<option value="{html_escape.escape(doc)}">{html_escape.escape(doc)}</option>' for doc in documents)}
        </select>
        
        <label>Label:</label>
        <select id="label-filter" onchange="applyFilters()">
            <option value="">All Labels</option>
            {''.join(f'<option value="{html_escape.escape(label)}">{html_escape.escape(label)}</option>' for label in labels)}
        </select>
        
        <label>Match Type:</label>
        <select id="match-filter" onchange="applyFilters()">
            <option value="">All Matches</option>
            <option value="✓ exact">Exact Match</option>
            <option value="≈ lenient">Lenient Match</option>
            <option value="✗ no match">No Match</option>
        </select>
    </div>
    
    <table id="comparison-table">
        <thead>
            <tr>
                <th>Document</th>
                <th>Label Name</th>
                <th>Human Annotation</th>
                <th>LLM Annotation</th>
                <th>Match Type</th>
                <th>Highlight Color</th>
                <th>Comment</th>
            </tr>
        </thead>
        <tbody id="table-body">
        </tbody>
    </table>
    
    <script>
        let data = {data_json};
        const STORAGE_KEY = 'annotation_comparison_data';
        
        // Color names mapping
        const COLOR_NAMES = {{
            0: '⚪ None',
            1: '🔴 Red',
            2: '🟢 Green',
            3: '🔵 Blue',
            4: '🟠 Orange',
            5: '🟣 Purple',
            6: '🟡 Yellow',
            7: '🔷 Cyan',
            8: '💗 Pink',
            9: '🟣 Indigo'
        }};
        
        // Load saved data from localStorage
        function loadSavedData() {{
            try {{
                const saved = localStorage.getItem(STORAGE_KEY);
                if (saved) {{
                    const savedData = JSON.parse(saved);
                    // Merge saved comments and colors with current data
                    savedData.forEach(savedRow => {{
                        const match = data.find(row => 
                            row.document === savedRow.document &&
                            row.label_name === savedRow.label_name &&
                            row.human_text === savedRow.human_text &&
                            row.llm_text === savedRow.llm_text
                        );
                        if (match) {{
                            match.comment = savedRow.comment || '';
                            match.rowColor = savedRow.rowColor || 'color-0';
                        }}
                    }});
                    showSaveStatus('Loaded saved data');
                }}
            }} catch (e) {{
                console.error('Error loading saved data:', e);
            }}
        }}
        
        // Save data to localStorage
        function saveData() {{
            try {{
                localStorage.setItem(STORAGE_KEY, JSON.stringify(data));
                showSaveStatus('Auto-saved ✓');
            }} catch (e) {{
                console.error('Error saving data:', e);
                showSaveStatus('Save failed!');
            }}
        }}
        
        function showSaveStatus(message) {{
            const status = document.getElementById('save-status');
            status.textContent = message;
            setTimeout(() => {{
                status.textContent = '';
            }}, 3000);
        }}
        
        // Export data as JSON file
        function exportData() {{
            const dataStr = JSON.stringify(data, null, 2);
            const dataBlob = new Blob([dataStr], {{ type: 'application/json' }});
            const url = URL.createObjectURL(dataBlob);
            const link = document.createElement('a');
            link.href = url;
            const timestamp = new Date().toISOString().replace(/[:.]/g, '-').slice(0, -5);
            link.download = `annotation_comparison_${{timestamp}}.json`;
            link.click();
            URL.revokeObjectURL(url);
            showSaveStatus('Exported successfully!');
        }}
        
        // Import data from JSON file
        function importData(event) {{
            const file = event.target.files[0];
            if (!file) return;
            
            const reader = new FileReader();
            reader.onload = function(e) {{
                try {{
                    const importedData = JSON.parse(e.target.result);
                    // Merge imported data with current data
                    importedData.forEach(importedRow => {{
                        const match = data.find(row => 
                            row.document === importedRow.document &&
                            row.label_name === importedRow.label_name &&
                            row.human_text === importedRow.human_text &&
                            row.llm_text === importedRow.llm_text
                        );
                        if (match) {{
                            match.comment = importedRow.comment || '';
                            match.rowColor = importedRow.rowColor || 'color-0';
                        }}
                    }});
                    saveData();
                    renderTable();
                    showSaveStatus('Import successful!');
                }} catch (e) {{
                    alert('Error importing file: ' + e.message);
                }}
            }};
            reader.readAsText(file);
            // Reset file input
            event.target.value = '';
        }}
        
        function escapeHtml(text) {{
            const div = document.createElement('div');
            div.textContent = text || '';
            return div.innerHTML;
        }}
        
        function getMatchClass(matchType) {{
            if (matchType === '✓ exact') return 'match-exact';
            if (matchType === '≈ lenient') return 'match-lenient';
            return 'match-none';
        }}
        
        function renderTable() {{
            const docFilter = document.getElementById('doc-filter').value;
            const labelFilter = document.getElementById('label-filter').value;
            const matchFilter = document.getElementById('match-filter').value;
            
            const filteredData = data.filter(row => {{
                const docMatch = !docFilter || row.document === docFilter;
                const labelMatch = !labelFilter || row.label_name === labelFilter;
                const matchTypeMatch = !matchFilter || row.match_type === matchFilter;
                return docMatch && labelMatch && matchTypeMatch;
            }});
            
            const tbody = document.getElementById('table-body');
            tbody.innerHTML = '';
            
            filteredData.forEach((row, displayIdx) => {{
                const dataIdx = data.indexOf(row);
                const tr = document.createElement('tr');
                tr.className = row.rowColor || 'color-0';
                
                // Create cells
                const cellDoc = document.createElement('td');
                cellDoc.textContent = row.document;
                
                const cellLabel = document.createElement('td');
                cellLabel.textContent = row.label_name;
                
                const cellHuman = document.createElement('td');
                cellHuman.textContent = row.human_text || '(no match)';
                
                const cellLLM = document.createElement('td');
                cellLLM.textContent = row.llm_text || '(no match)';
                
                const cellMatch = document.createElement('td');
                cellMatch.textContent = row.match_type;
                cellMatch.className = getMatchClass(row.match_type);
                
                const cellColor = document.createElement('td');
                const colorSelect = document.createElement('select');
                for (let i = 0; i <= 9; i++) {{
                    const option = document.createElement('option');
                    option.value = i;
                    option.textContent = COLOR_NAMES[i] || ('Color ' + i);
                    if (row.rowColor === 'color-' + i) {{
                        option.selected = true;
                    }}
                    colorSelect.appendChild(option);
                }}
                colorSelect.onchange = function() {{
                    changeRowColor(dataIdx, this.value);
                }};
                cellColor.appendChild(colorSelect);
                
                const cellComment = document.createElement('td');
                const commentTextarea = document.createElement('textarea');
                commentTextarea.className = 'comment-input';
                commentTextarea.value = row.comment || '';
                commentTextarea.onchange = function() {{
                    updateComment(dataIdx, this.value);
                }};
                cellComment.appendChild(commentTextarea);
                
                // Append all cells to row
                tr.appendChild(cellDoc);
                tr.appendChild(cellLabel);
                tr.appendChild(cellHuman);
                tr.appendChild(cellLLM);
                tr.appendChild(cellMatch);
                tr.appendChild(cellColor);
                tr.appendChild(cellComment);
                
                tbody.appendChild(tr);
            }});
            
            document.getElementById('visible-count').textContent = filteredData.length;
            document.getElementById('total-count').textContent = data.length;
        }}
        
        function applyFilters() {{
            renderTable();
        }}
        
        function changeRowColor(dataIdx, colorIdx) {{
            if (data[dataIdx]) {{
                data[dataIdx].rowColor = 'color-' + colorIdx;
                saveData();
                renderTable();
            }}
        }}
        
        function updateComment(dataIdx, value) {{
            if (data[dataIdx]) {{
                data[dataIdx].comment = value;
                saveData();
            }}
        }}
        
        // Initial load and render
        if (document.readyState === 'loading') {{
            document.addEventListener('DOMContentLoaded', function() {{
                loadSavedData();
                renderTable();
            }});
        }} else {{
            loadSavedData();
            renderTable();
        }}
    </script>
</body>
</html>
"""
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"HTML report generated: {output_path}")
    return output_path

In [74]:
# Generate the interactive HTML report
output_html_path = fr"{project_root}\annotation_comparison_report.html"
generate_html_report(results, output_html_path)

HTML report generated: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\annotation_comparison_report.html


'C:\\Users\\zakga\\OneDrive\\Documents\\code\\LeREaD_annotation_process\\annotation_comparison_report.html'